In [1]:
import pandas as pd

In [2]:
df_l = pd.read_csv('hospital_account_info.csv')
df_r = pd.read_csv('hospital_reimbursement.csv')

In [3]:
df_l = df_l.rename(columns={"Account_Num":"unique_id", "Facility Name":"Facility_Name"})
df_r = df_r.rename(columns={"Provider_Num":"unique_id"})
df_r = df_r.rename(columns={"Provider Name":"Facility_Name", "Provider Street Address":"Address", "Provider City":"City", "Provider State":"State"})
df_l = df_l.drop(columns={'ZIP Code', 'County Name', 'Phone Number', 'Hospital Type', 'Hospital Ownership'})
df_r = df_r.drop(columns={'Provider Zip Code', 'Total Discharges', 'Average Covered Charges', 'Average Total Payments', 'Average Medicare Payments'})

In [4]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl

In [5]:
settings = {"link_type":"link_only"}
linker = DuckDBLinker([df_l, df_r], settings, input_table_aliases=["df_left", "df_right"])

In [39]:
blocking_rule_1 = block_on(["City", "State"])
blocking_rule_2 = block_on(["Facility_Name"])
blocking_rule_3 = block_on(["Address"])

In [40]:
count = linker.count_num_comparisons_from_blocking_rule(blocking_rule_3)
print(f"Number of comparisons generated by '{blocking_rule_3.blocking_rule_sql}': {count:,.0f}")

Number of comparisons generated by 'l."Address" = r."Address"': 2,586


In [41]:
blocking_rules = [blocking_rule_1, blocking_rule_2, blocking_rule_3]
linker.cumulative_num_comparisons_from_blocking_rules_chart(blocking_rules)

alt.Chart(...)

In [74]:
settings = {
    "link_type": "link_only",
    "blocking_rules_to_generate_predictions": [
        block_on(["State","City"]),
        block_on(["Facility_Name"]),
        block_on(["Address"])
    ],
    "comparisons": [
        ctl.name_comparison("Facility_Name"),
        cl.jaccard_at_thresholds("Address", 0.9),
        cl.exact_match("State", term_frequency_adjustments=True),
        cl.exact_match("City", term_frequency_adjustments=True)
    ],
    "retain_matching_columns" : True,
    "retain_intermediate_calculation_columns": True
}

linker = DuckDBLinker([df_l, df_r], settings, input_table_aliases=["df_left", "df_right"])

In [75]:
deterministic_rules = [
    "l.City = r.City and jaccard(r.Address, l.Address) >= 0.9",
    "l.City = r.City and jaccard(r.Facility_Name, l.Facility_Name) >= 0.9",
    "l.Facility_Name = r.Facility_Name and jaccard(r.Address, l.Address) >= 0.9",
    "l.Address = r.Address and jaccard(r.Facility_Name, l.Facility_Name) >= 0.9",
]
linker.estimate_probability_two_random_records_match(deterministic_rules, recall=0.6)

Probability two random records match is estimated to be  0.00032.
This means that amongst all possible pairwise record comparisons, one in 3,129.15 are expected to match.  With 14,399,283 total possible comparisons, we expect a total of around 4,601.67 matching pairs


In [76]:
linker.estimate_u_using_random_sampling(max_pairs=1e6, seed=1)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - Facility_Name (no m values are trained).
    - Address (no m values are trained).
    - State (no m values are trained).
    - City (no m values are trained).


In [77]:
session_facility_name = linker.estimate_parameters_using_expectation_maximisation(block_on("Facility_Name"))
session_address = linker.estimate_parameters_using_expectation_maximisation(block_on("Address"))
session_city_state = linker.estimate_parameters_using_expectation_maximisation(block_on(["City","State"]))


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."Facility_Name" = r."Facility_Name"

Parameter estimates will be made for the following comparison(s):
    - Address
    - State
    - City

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - Facility_Name

Iteration 1: Largest change in params was 0.818 in probability_two_random_records_match
Iteration 2: Largest change in params was 0.0139 in the m_probability of Address, level `All other comparisons`
Iteration 3: Largest change in params was 2.1e-05 in the m_probability of Address, level `All other comparisons`

EM converged after 3 iterations

Your model is not yet fully trained. Missing estimates for:
    - Facility_Name (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."Address" = r."Address"

Parameter estimates will be made for t

In [78]:
linker.match_weights_chart()

alt.VConcatChart(...)

In [79]:
linker.m_u_parameters_chart()

alt.HConcatChart(...)

In [80]:
results = linker.predict(threshold_match_probability=0.8)
results.as_pandas_dataframe(limit=5)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,Facility_Name_l,Facility_Name_r,gamma_Facility_Name,bf_Facility_Name,...,bf_State,bf_tf_adj_State,City_l,City_r,gamma_City,tf_City_l,tf_City_r,bf_City,bf_tf_adj_City,match_key
0,2.037183,0.804092,df_left,df_right,72779,168047,ANDERSON RMC SOUTH,ANDERSON REGIONAL MEDICAL CTR,1,5.306021,...,29.019839,1.871041,MERIDIAN,MERIDIAN,1,0.000747,0.000747,1164.377067,1.150257,0
1,2.070048,0.807656,df_left,df_right,58001,145995,CONWAY BEHAVIORAL HEALTH,CONWAY REGIONAL MEDICAL CENTER,1,5.306021,...,29.019839,2.233177,CONWAY,CONWAY,1,0.000871,0.000871,1164.377067,0.985935,0
2,2.143349,0.815425,df_left,df_right,10199,104376,SENTARA PRINCESS ANNE HOSPITAL,SENTARA VIRGINIA BEACH GENERAL HOSPITAL,1,5.306021,...,29.019839,1.678267,VIRGINIA BEACH,VIRGINIA BEACH,1,0.000622,0.000622,1164.377067,1.380309,0
3,2.143349,0.815425,df_left,df_right,11155,506708,SENTARA VIRGINIA BEACH GENERAL HOSPITAL,SENTARA PRINCESS ANNE HOSPITAL,1,5.306021,...,29.019839,1.678267,VIRGINIA BEACH,VIRGINIA BEACH,1,0.000622,0.000622,1164.377067,1.380309,0
4,2.249000,0.826194,df_left,df_right,34909,704142,KAISER FOUNDATION HOSPITAL - SACRAMENTO,KAISER FOUNDATION HOSP SO SACRAMENTO,2,64.046532,...,29.019839,0.448807,SACRAMENTO,SACRAMENTO,1,0.001867,0.001867,1164.377067,0.460103,0


In [81]:
records_to_view  = results.as_record_dict(limit=5)
linker.waterfall_chart(records_to_view, filter_nulls=False)

alt.LayerChart(...)

In [82]:
linker.comparison_viewer_dashboard(results, "link_only.html", overwrite=True)